# Wool Dynamics Analysis

Four complementary views of match tempo and objective play:

- **Section 1** — Per-Wool Node Coverage: defenders vs. attackers at each wool room node over match time.
- **Section 2** — Y-Level Phase Detection: ground play vs. skybridge phase per team.
- **Section 3** — Per-Wool Attack Depth: independent pressure series for each wool objective.
- **Section 4** — Carry Chain Timeline: reconstructed wool carry attempts from touch to capture/loss.

---

**Background:** In Capture the Wool each team has two wool objectives in the enemy's base. A carrier
touches the wool, transports it across the map, and places it on their monument to score.
Defenders cannot enter their own wool rooms — they guard the *entrance corridor* from outside.
Teams typically concentrate defence on one wool, leaving the other structurally exposed.

In [ ]:
import os, sys
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

import json
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import duckdb
from pathlib import Path

DB_PATH    = Path('match_analysis/metadata.db')
OUTPUT_DIR = Path('output')
OUTPUT_DIR.mkdir(exist_ok=True)

# ── Shared tunables ──────────────────────────────────────────────────────────
INSPECT_MATCH_ID   = 3      # tumbleweed
BUCKET_S           = 60     # time bucket for coverage / depth charts
SKYBRIDGE_Y        = 22     # Y >= this → elevated / skybridge
CARRY_WAVE_GAP_S   = 120    # gap between touches to split carry waves

_MINECRAFT_COLORS = {
    'red': '#FF5555', 'dark_red': '#AA0000',
    'blue': '#5555FF', 'dark_blue': '#0000AA',
    'green': '#55FF55', 'dark_green': '#00AA00',
    'yellow': '#DDDD00', 'gold': '#FFAA00',
    'aqua': '#55FFFF', 'dark_aqua': '#00AAAA',
    'purple': '#AA00AA', 'light_purple': '#FF55FF',
    'white': '#FFFFFF', 'gray': '#AAAAAA',
    'dark_gray': '#555555', 'black': '#000000',
    'orange': '#FFA500', 'lime': '#00CC44',
    'cyan': '#00CCCC',
}

def team_hex(color_raw: str) -> str:
    key = color_raw.lower().replace(' ', '_')
    if key in _MINECRAFT_COLORS:
        return _MINECRAFT_COLORS[key]
    try:
        return matplotlib.colors.to_hex(color_raw)
    except ValueError:
        return '#AAAAAA'

def open_db(read_only=True):
    return duckdb.connect(str(DB_PATH), read_only=read_only)

print('Ready.')

In [ ]:
# ── Load match metadata and map topology ─────────────────────────────────────
conn = open_db()

meta = conn.execute("""
    SELECT mat.match_id, mat.map_id, mat.match_duration, m.map_slug, m.max_build_height
    FROM matches mat JOIN maps m ON mat.map_id = m.map_id
    WHERE mat.match_id = ?
""", [INSPECT_MATCH_ID]).fetchone()
match_id, map_id, match_duration_s, map_slug, max_build_h = meta

team_colors = {}
for team_raw, color_raw in conn.execute(
    "SELECT team, team_color FROM map_spawns WHERE map_id = ?", [map_id]
).fetchall():
    team = team_raw.removesuffix('-team')
    team_colors[team] = team_hex(color_raw)

baselines = conn.execute("""
    SELECT team, wool_id, wool_x, wool_z, baseline_distance
    FROM wool_spawn_baselines WHERE map_id = ?
""", [map_id]).df()

wool_events_all = conn.execute("""
    SELECT we.timestamp, we.event_type, we.player_id,
           we.wool_id, we.x, we.y, we.z, pts.team
    FROM wool_events we
    LEFT JOIN player_team_segments pts
        ON pts.player_id = we.player_id AND pts.match_id = we.match_id
        AND pts.start_timestamp <= we.timestamp
        AND (pts.end_timestamp IS NULL OR pts.end_timestamp >= we.timestamp)
    WHERE we.match_id = ?
    ORDER BY we.wool_id, we.timestamp
""", [match_id]).df()

conn.close()

# Load map graph for node annotations
with open(f'output/{map_slug}/map_graph.json') as f:
    mg = json.load(f)
with open(f'output/{map_slug}/map_data.json') as f:
    md = json.load(f)

# Build wool_nodes dict from map_graph skeleton nodes
wool_nodes: dict[str, dict] = {}
for n in mg['map_graph']['nodes']:
    if n['poi_type'] == 'wool':
        color = n['poi_color']
        wool_nodes[color] = {
            'map_node_id':    n['map_node_id'],
            'defending_team': n['team'],
            'coords':         n['coords'],
        }

# Derive attacking_team directly (the team that is NOT the defender)
all_teams = sorted(team_colors.keys())
for color, info in wool_nodes.items():
    def_team = info['defending_team']
    atk_candidates = [t for t in all_teams if t != def_team]
    info['attacking_team'] = atk_candidates[0] if len(atk_candidates) == 1 else None

# Match wool_id from wool_events first-touch coordinates.
# The wool_id numbering in wool_events can differ from wool_spawn_baselines,
# so we match by spatial proximity to the skeleton node rather than by ID.
if len(wool_events_all) > 0:
    first_touches = (
        wool_events_all[wool_events_all.event_type == 6]
        .sort_values('timestamp')
        .groupby('wool_id')
        .first()
        .reset_index()
    )
    for color, info in wool_nodes.items():
        cx, cz = info['coords']
        best_row, best_dist = None, float('inf')
        for _, row in first_touches.iterrows():
            dist = ((row.x - cx) ** 2 + (row.z - cz) ** 2) ** 0.5
            if dist < best_dist:
                best_dist = dist
                best_row = row
        if best_row is not None and best_dist < 20:
            info['wool_id'] = int(best_row.wool_id)

# Match baseline distance by coordinate proximity (generous tolerance)
for color, info in wool_nodes.items():
    cx, cz = info['coords']
    best_row, best_dist = None, float('inf')
    for _, row in baselines.iterrows():
        dist = ((row.wool_x - cx) ** 2 + (row.wool_z - cz) ** 2) ** 0.5
        if dist < best_dist:
            best_dist = dist
            best_row = row
    if best_row is not None and best_dist < 20:
        info['baseline'] = float(best_row.baseline_distance)

print(f'Match {match_id}: {map_slug}  duration={match_duration_s:.0f}s  max_build_h={max_build_h}')
print('Team colours:', team_colors)
print('Wool nodes:')
for c, v in wool_nodes.items():
    print(f'  {c:10s}  node={v["map_node_id"]}  def={v["defending_team"]}  atk={v.get("attacking_team")}  wool_id={v.get("wool_id")}  baseline={v.get("baseline")}')

---
## Section 1 — Per-Wool Node Coverage

For each of the four wool objectives, count friendly (defending) and enemy (attacking)
player ticks at the wool's skeleton node per 60-second bucket.  A sustained defender
presence suppresses captures; a gap in coverage is a vulnerability.  Wool touch events
are overlaid as vertical markers — △ = touch, ★ = capture.

In [ ]:
conn = open_db()

# Per-node per-bucket team presence
presence_df = conn.execute("""
    SELECT
        pe.nearest_graph_node,
        (pe.timestamp // ?) * ? AS bucket,
        pts.team,
        COUNT(*) AS ticks
    FROM position_events pe
    LEFT JOIN player_team_segments pts
        ON pts.player_id = pe.player_id AND pts.match_id = pe.match_id
        AND pts.start_timestamp <= pe.timestamp
        AND (pts.end_timestamp IS NULL OR pts.end_timestamp >= pe.timestamp)
    WHERE pe.match_id = ? AND pe.nearest_graph_node IS NOT NULL
    GROUP BY pe.nearest_graph_node, bucket, pts.team
""", [BUCKET_S, BUCKET_S, match_id]).df()

conn.close()

n_wools = len(wool_nodes)
fig, axes = plt.subplots(n_wools, 1, figsize=(16, 4 * n_wools),
                          sharex=True, constrained_layout=True)
if n_wools == 1:
    axes = [axes]

xs_full = np.arange(0, match_duration_s + BUCKET_S, BUCKET_S)

for ax, (color, info) in zip(axes, sorted(wool_nodes.items())):
    node_id = info['map_node_id']
    def_team = info['defending_team']
    atk_team = info.get('attacking_team')
    wool_id  = info.get('wool_id')
    def_color = team_colors.get(def_team, '#888888')
    atk_color = team_colors.get(atk_team, '#AAAAAA') if atk_team else '#AAAAAA'
    wool_hex  = _MINECRAFT_COLORS.get(color, '#888888')

    sub = presence_df[presence_df.nearest_graph_node == node_id]
    def_ticks = sub[sub.team == def_team].set_index('bucket')['ticks'].reindex(xs_full, fill_value=0)
    atk_ticks = sub[sub.team == atk_team].set_index('bucket')['ticks'].reindex(xs_full, fill_value=0) if atk_team else pd.Series(0, index=xs_full)

    ax.fill_between(xs_full, def_ticks, alpha=0.35, color=def_color, step='post', label=f'{def_team} (defenders)')
    ax.fill_between(xs_full, atk_ticks, alpha=0.35, color=atk_color, step='post', label=f'{atk_team} (attackers)')
    ax.plot(xs_full, def_ticks, color=def_color, lw=1.5, drawstyle='steps-post')
    ax.plot(xs_full, atk_ticks, color=atk_color, lw=1.5, drawstyle='steps-post')

    # Wool events
    if wool_id is not None:
        touches  = wool_events_all[(wool_events_all.wool_id == wool_id) & (wool_events_all.event_type == 6)]
        captures = wool_events_all[(wool_events_all.wool_id == wool_id) & (wool_events_all.event_type == 7)]
        ymax = ax.get_ylim()[1] if ax.get_ylim()[1] > 0 else 1
        ax.vlines(touches.timestamp,  0, ax.get_ylim()[1] or 50, color=wool_hex, lw=0.8, alpha=0.5, label='touch')
        ax.scatter(captures.timestamp, [ax.get_ylim()[1] * 0.9 or 45] * len(captures),
                   marker='*', color=wool_hex, s=150, zorder=5, label='capture')

    ax.set_xlim(0, match_duration_s)
    ax.set_ylabel('Ticks at node')
    ax.set_title(f'{color.capitalize()} wool  [{def_team} defends | {atk_team} attacks]  node={node_id}')
    ax.legend(fontsize=8, loc='upper right')

axes[-1].set_xlabel('Elapsed seconds')
fig.suptitle(f'{map_slug} — Wool node coverage per {BUCKET_S}s bucket', fontsize=13)
out = OUTPUT_DIR / f'wool_coverage_{map_slug}.png'
fig.savefig(out, dpi=130, bbox_inches='tight')
plt.show()
print(f'Saved: {out}')

---
## Section 2 — Y-Level Phase Detection

Rolling 5-minute median Y per team reveals when teams transition from ground-level play
to skybridge-level play.  The skybridge (built at `max_build_height`) creates a second
traffic layer above the map — once established it becomes the fastest route to the enemy base.
An upward shift in the team's median Y marks the start of the skybridge phase.

The right panel shows the Y distribution early vs. late to confirm bimodality.

In [ ]:
conn = open_db()

Y_BUCKET_S = 300   # 5-minute rolling window

y_ts = conn.execute("""
    SELECT
        (pe.timestamp // ?) * ? AS bucket,
        pts.team,
        PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY pe.y) AS median_y,
        AVG(CASE WHEN pe.y >= ? THEN 1.0 ELSE 0.0 END)::FLOAT AS frac_elevated,
        COUNT(*) AS n
    FROM position_events pe
    LEFT JOIN player_team_segments pts
        ON pts.player_id = pe.player_id AND pts.match_id = pe.match_id
        AND pts.start_timestamp <= pe.timestamp
        AND (pts.end_timestamp IS NULL OR pts.end_timestamp >= pe.timestamp)
    WHERE pe.match_id = ? AND pts.team IS NOT NULL
    GROUP BY bucket, pts.team
    ORDER BY bucket, pts.team
""", [Y_BUCKET_S, Y_BUCKET_S, SKYBRIDGE_Y, match_id]).df()

# Early (<600s) and late (>match_duration-600s) Y histograms
y_early = conn.execute("""
    SELECT y FROM position_events WHERE match_id = ? AND timestamp < 600 AND y >= 0
""", [match_id]).df()
y_late = conn.execute("""
    SELECT y FROM position_events WHERE match_id = ? AND timestamp > ? AND y >= 0
""", [match_id, match_duration_s - 600]).df()

conn.close()

teams_y = sorted(y_ts['team'].dropna().unique())
fig, (ax_ts, ax_hist) = plt.subplots(1, 2, figsize=(16, 5), constrained_layout=True)

# ── Left: rolling median Y ──────────────────────────────────────────────────
for team in teams_y:
    color = team_colors.get(team, '#888888')
    tm = y_ts[y_ts.team == team].sort_values('bucket')
    ax_ts.plot(tm.bucket, tm.median_y, lw=2, color=color, label=f'{team} median Y')
    ax_ts.fill_between(tm.bucket,
                        tm.median_y - 2, tm.median_y + 2,
                        color=color, alpha=0.12)

ax_ts.axhline(SKYBRIDGE_Y, color='grey', ls='--', lw=1, label=f'threshold Y={SKYBRIDGE_Y}')
if max_build_h:
    ax_ts.axhline(max_build_h, color='black', ls=':', lw=1, label=f'max build Y={max_build_h}')
ax_ts.set_xlabel(f'Elapsed seconds (5-min buckets)')
ax_ts.set_ylabel('Median player Y')
ax_ts.set_title(f'{map_slug} — Y-level phase over match time')
ax_ts.legend(fontsize=9)
ax_ts.set_xlim(0, match_duration_s)

# ── Right: Y distribution histograms early vs late ──────────────────────────
bins = np.arange(0, max_build_h + 3 if max_build_h else 35, 1)
ax_hist.hist(y_early.y, bins=bins, alpha=0.55, color='steelblue', label='Early (<10 min)', density=True)
ax_hist.hist(y_late.y,  bins=bins, alpha=0.55, color='tomato',    label='Late (last 10 min)', density=True)
ax_hist.axvline(SKYBRIDGE_Y, color='grey', ls='--', lw=1)
ax_hist.set_xlabel('Player Y')
ax_hist.set_ylabel('Density')
ax_hist.set_title('Y distribution: early vs. late game')
ax_hist.legend(fontsize=9)

fig.suptitle(f'{map_slug} — Skybridge phase detection', fontsize=13)
out = OUTPUT_DIR / f'y_phase_{map_slug}.png'
fig.savefig(out, dpi=130, bbox_inches='tight')
plt.show()
print(f'Saved: {out}')

---
## Section 3 — Per-Wool Attack Depth

The unified `attack_depth` in `life_segment_features` takes the maximum over all enemy wools,
which obscures *which* objective is under pressure.  Here we compute independent depth series
for each wool: `depth_W(t) = max(0, 1 − dist(player, wool_W) / baseline_W)` averaged over
all players of the attacking team per time bucket.

Divergence between the two depth series for the same attacking team reveals the
"forced single defence" dynamic: one wool sustains sustained pressure while the other is quiet.

In [ ]:
conn = open_db()

# Load all positions with team labels
pos_df = conn.execute("""
    SELECT pe.timestamp, pe.player_id, pe.x, pe.z, pts.team
    FROM position_events pe
    LEFT JOIN player_team_segments pts
        ON pts.player_id = pe.player_id AND pts.match_id = pe.match_id
        AND pts.start_timestamp <= pe.timestamp
        AND (pts.end_timestamp IS NULL OR pts.end_timestamp >= pe.timestamp)
    WHERE pe.match_id = ? AND pts.team IS NOT NULL
""", [match_id]).df()
conn.close()

# Compute per-wool attack depth per time bucket per attacking team
records = []
for color, info in wool_nodes.items():
    atk_team = info.get('attacking_team')
    baseline = info.get('baseline')
    wx, wz = info['coords']
    if atk_team is None or baseline is None or baseline == 0:
        continue
    atk_pos = pos_df[pos_df.team == atk_team].copy()
    dist = np.sqrt((atk_pos.x - wx)**2 + (atk_pos.z - wz)**2)
    atk_pos = atk_pos.assign(depth=np.clip(1.0 - dist / baseline, 0.0, 1.0))
    atk_pos['bucket'] = (atk_pos.timestamp // BUCKET_S) * BUCKET_S
    agg = atk_pos.groupby('bucket')['depth'].mean().reset_index()
    agg['wool_color'] = color
    agg['attacking_team'] = atk_team
    records.append(agg)

depth_df = pd.concat(records, ignore_index=True) if records else pd.DataFrame()

# One panel per attacking team
atk_teams = sorted(depth_df['attacking_team'].unique()) if not depth_df.empty else []
fig, axes = plt.subplots(len(atk_teams), 1, figsize=(16, 4.5 * len(atk_teams)),
                          sharex=True, constrained_layout=True)
if len(atk_teams) == 1:
    axes = [axes]

for ax, atk_team in zip(axes, atk_teams):
    sub = depth_df[depth_df.attacking_team == atk_team]
    for color, grp in sub.groupby('wool_color'):
        wool_hex = _MINECRAFT_COLORS.get(color, '#888888')
        xs = grp.sort_values('bucket').bucket.values
        ys = grp.sort_values('bucket').depth.values
        ax.plot(xs, ys, lw=2, color=wool_hex, label=f'{color} wool')

        # Mark touches and captures for this wool
        wid = next((v.get('wool_id') for c, v in wool_nodes.items() if c == color), None)
        if wid is not None:
            touches  = wool_events_all[(wool_events_all.wool_id == wid) & (wool_events_all.event_type == 6)]
            captures = wool_events_all[(wool_events_all.wool_id == wid) & (wool_events_all.event_type == 7)]
            ax.scatter(touches.timestamp,  [0.02] * len(touches),  marker='^', color=wool_hex, s=50, alpha=0.6, zorder=4)
            ax.scatter(captures.timestamp, [0.02] * len(captures), marker='*', color=wool_hex, s=150, zorder=5)

    tc = team_colors.get(atk_team, '#888')
    ax.set_facecolor((*matplotlib.colors.to_rgb(tc), 0.04))
    ax.set_ylabel('Mean attack depth')
    ax.set_ylim(0, 1.05)
    ax.set_xlim(0, match_duration_s)
    ax.set_title(f'{atk_team} team — per-wool attack depth  (△ = touch · ★ = capture)')
    ax.legend(fontsize=9)

axes[-1].set_xlabel('Elapsed seconds')
fig.suptitle(f'{map_slug} — Per-wool attack depth', fontsize=13)
out = OUTPUT_DIR / f'wool_depth_{map_slug}.png'
fig.savefig(out, dpi=130, bbox_inches='tight')
plt.show()
print(f'Saved: {out}')

---
## Section 4 — Carry Chain Timeline

Each wool carry "wave" is reconstructed from touch events: a burst of touches on the same
wool_id (separated by at most `CARRY_WAVE_GAP_S` seconds) forms one wave.  The bar spans
from first touch to capture/loss, coloured by the attacking team.  Handoffs (wool changing
hands mid-carry) are annotated.  The approach type — **ground** (Y < threshold before touch)
vs. **skybridge** (Y ≥ threshold) — tags each wave to distinguish early-game and late-game
push strategies.

In [ ]:
conn = open_db()

# Try to load pre-computed carry chains; fall back to on-the-fly reconstruction
try:
    chains_df = conn.execute("""
        SELECT wcc.*, wsb.team AS attacking_team_chk
        FROM wool_carry_chains wcc
        LEFT JOIN wool_spawn_baselines wsb
            ON wsb.map_id = (SELECT map_id FROM matches WHERE match_id = wcc.match_id)
            AND wsb.wool_id = wcc.wool_id
        WHERE wcc.match_id = ?
        ORDER BY wcc.wool_id, wcc.start_timestamp
    """, [match_id]).df()
    pre_computed = len(chains_df) > 0
except Exception:
    pre_computed = False
    chains_df = pd.DataFrame()

death_df = conn.execute("""
    SELECT timestamp, player_id, y FROM combat_events
    WHERE match_id = ? AND event_type = 4
    ORDER BY timestamp
""", [match_id]).df()

conn.close()

# ── On-the-fly reconstruction if DB table not yet populated ──────────────────
def reconstruct_chains(wool_events, death_df, pos_df, match_duration_s, gap_s=CARRY_WAVE_GAP_S):
    deaths_by_player = {}
    for _, row in death_df.iterrows():
        deaths_by_player.setdefault(row.player_id, []).append((row.timestamp, row.y))

    chains = []
    for wool_id, events in wool_events.groupby('wool_id'):
        touches  = events[events.event_type == 6].sort_values('timestamp')
        captures = events[events.event_type == 7]
        cap_ts   = set(captures.timestamp.tolist())

        waves = []
        cur = []
        for _, t in touches.iterrows():
            if not cur or t.timestamp - cur[-1]['ts'] <= gap_s:
                cur.append({'ts': t.timestamp, 'pid': t.player_id, 'team': t.team,
                            'x': t.x, 'y': t.y, 'z': t.z})
            else:
                waves.append(cur); cur = [dict(ts=t.timestamp, pid=t.player_id,
                                               team=t.team, x=t.x, y=t.y, z=t.z)]
        if cur:
            waves.append(cur)

        for wi, wave in enumerate(waves):
            first = wave[0]
            last  = wave[-1]
            carriers = []
            for w in wave:
                if not carriers or w['pid'] != carriers[-1]:
                    carriers.append(w['pid'])
            n_handoffs = len(carriers) - 1

            # Approach: max Y of first carrier in 60s before touch
            pre = pos_df[(pos_df.player_id == first['pid']) &
                         (pos_df.timestamp >= first['ts'] - 60) &
                         (pos_df.timestamp <= first['ts'])]
            max_y_pre = int(pre.y.max()) if len(pre) > 0 and 'y' in pre.columns else None
            approach = ('skybridge' if max_y_pre is not None and max_y_pre >= SKYBRIDGE_Y
                        else 'ground' if max_y_pre is not None else 'unknown')

            outcome = 'incomplete'
            end_ts  = last['ts']
            relevant_caps = [c for c in cap_ts if c >= first['ts']]
            if relevant_caps:
                end_ts  = min(relevant_caps)
                outcome = 'captured'
            else:
                for d_ts, d_y in deaths_by_player.get(carriers[-1], []):
                    if d_ts >= first['ts']:
                        end_ts  = d_ts
                        outcome = 'dropped_void' if d_y < 0 else 'dropped_land'
                        break

            chains.append(dict(
                wool_id=wool_id, wave_idx=wi,
                attacking_team=first['team'],
                n_handoffs=n_handoffs, n_carriers=len(set(carriers)),
                start_timestamp=first['ts'], end_timestamp=end_ts,
                duration_s=float(end_ts - first['ts']),
                outcome=outcome,
                first_y=first['y'], max_y_before_touch=max_y_pre,
                approach_type=approach,
            ))
    return pd.DataFrame(chains)

if not pre_computed:
    pos_for_chains = conn_ro = duckdb.connect(str(DB_PATH), read_only=True).execute(
        "SELECT timestamp, player_id, x, y, z FROM position_events WHERE match_id = ?",
        [match_id]).df()
    chains_df = reconstruct_chains(wool_events_all, death_df, pos_for_chains, match_duration_s)
    print('Chains reconstructed on-the-fly')
else:
    print('Using pre-computed carry chains from DB')

print(chains_df[['wool_id','wave_idx','attacking_team','n_handoffs','duration_s','outcome','approach_type']].to_string())

In [ ]:
# ── Gantt-style carry timeline ───────────────────────────────────────────────
wool_id_to_color = {v.get('wool_id'): k for k, v in wool_nodes.items() if v.get('wool_id')}
wool_id_to_def   = {v.get('wool_id'): v['defending_team'] for v in wool_nodes.values() if v.get('wool_id')}

wool_ids = sorted(chains_df.wool_id.unique()) if len(chains_df) else []

fig, ax = plt.subplots(figsize=(18, max(3, len(wool_ids) * 1.6)), constrained_layout=True)

outcome_hatch = {'captured': '',  'dropped_land': '//',  'dropped_void': 'xx',  'incomplete': '..'}
outcome_alpha = {'captured': 0.90, 'dropped_land': 0.50, 'dropped_void': 0.35, 'incomplete': 0.40}

y_tick_labels = []
for yi, wid in enumerate(wool_ids):
    wcolor_name = wool_id_to_color.get(wid, 'unknown')
    def_team    = wool_id_to_def.get(wid, '?')
    bar_y       = yi

    sub = chains_df[chains_df.wool_id == wid].sort_values('start_timestamp')
    for _, row in sub.iterrows():
        atk_team = row.get('attacking_team') or '?'
        bar_color = team_colors.get(atk_team, '#AAAAAA')
        start = row.start_timestamp
        end   = row.end_timestamp if pd.notna(row.end_timestamp) else match_duration_s
        width = max(end - start, 10)
        alpha  = outcome_alpha.get(row.outcome, 0.5)
        hatch  = outcome_hatch.get(row.outcome, '')

        ax.barh(bar_y, width, left=start, height=0.55,
                color=bar_color, alpha=alpha, hatch=hatch,
                edgecolor='white', linewidth=0.5)

        # Annotate handoffs and approach type
        label_parts = []
        if row.n_handoffs > 0:
            label_parts.append(f'+{int(row.n_handoffs)}↔')
        approach = row.get('approach_type') or ''
        if approach == 'skybridge':
            label_parts.append('sky')
        elif approach == 'ground':
            label_parts.append('gnd')
        if label_parts:
            ax.text(start + width / 2, bar_y, ' '.join(label_parts),
                    ha='center', va='center', fontsize=7, color='white',
                    fontweight='bold', clip_on=True)

        if row.outcome == 'captured':
            ax.scatter([end], [bar_y], marker='*', color='gold', s=120, zorder=6)
        elif row.outcome == 'dropped_void':
            ax.scatter([end], [bar_y], marker='x', color='black', s=80, zorder=6)

    y_tick_labels.append(f'{wcolor_name} wool\n[{def_team} defends]  wool_id={wid}')

ax.set_yticks(range(len(wool_ids)))
ax.set_yticklabels(y_tick_labels, fontsize=9)
ax.set_xlim(0, match_duration_s)
ax.set_xlabel('Elapsed seconds')
ax.set_title(f'{map_slug} — Wool carry chain timeline  (★ = capture · ✕ = void loss)', fontsize=12)

# Legend
legend_elements = [
    mpatches.Patch(color='grey', alpha=0.9, label='captured'),
    mpatches.Patch(color='grey', alpha=0.5, hatch='//', label='dropped (land)'),
    mpatches.Patch(color='grey', alpha=0.35, hatch='xx', label='dropped (void)'),
    mpatches.Patch(color='grey', alpha=0.40, hatch='..', label='incomplete'),
]
for team, color in team_colors.items():
    legend_elements.append(mpatches.Patch(color=color, label=f'{team} carrier'))
ax.legend(handles=legend_elements, loc='upper right', fontsize=8, ncol=2)

out = OUTPUT_DIR / f'wool_carry_chains_{map_slug}.png'
fig.savefig(out, dpi=130, bbox_inches='tight')
plt.show()
print(f'Saved: {out}')

### Carry chain statistics

In [ ]:
# Summary table
if len(chains_df) > 0:
    summary = chains_df.groupby(['wool_id', 'attacking_team']).agg(
        waves        =('wave_idx', 'count'),
        captures     =('outcome', lambda x: (x == 'captured').sum()),
        handoffs     =('n_handoffs', 'sum'),
        sky_approaches=('approach_type', lambda x: (x == 'skybridge').sum()),
        gnd_approaches=('approach_type', lambda x: (x == 'ground').sum()),
        avg_duration_s=('duration_s', 'mean'),
    ).reset_index()
    summary['wool_color'] = summary['wool_id'].map(wool_id_to_color)
    print(summary.to_string(index=False))